# Bigtable Change Stream Data Source for PySpark — Examples

This notebook demonstrates how to use the Bigtable Change Stream data source:
- **Streaming Read** — consume Bigtable Change Streams as a Spark Structured Streaming source
- **Filter by Mutation Type** — process only SET_CELL, DELETE_ROW, etc.
- **Write to Delta Lake** — persist change stream events to Delta
- **Watermark Support** — use low_watermark for late data handling
- **Reconstruct full records** — use `transformWithState` to rebuild the latest row state per key

## Setup

Install the wheel (if running outside Databricks):
```bash
pip install dist/bigtable_data_source-0.1.0-py3-none-any.whl
```

In [ ]:
from pyspark.sql import SparkSession
from bigtable_data_source import BigtableChangeStreamDataSource

spark = SparkSession.builder.appName("bigtable-examples").getOrCreate()
spark.dataSource.register(BigtableChangeStreamDataSource)

In [ ]:
# Common options — update these for your environment
common_options = {
    "project_id": "<your-gcp-project>",
    "instance_id": "<your-bigtable-instance>",
    "table_id": "<your-table>",
}

---
## Basic Streaming Read

Read all change stream events from the Bigtable table.

In [ ]:
changes = spark.readStream \
    .format("bigtable_changes") \
    .options(**common_options) \
    .load()

changes.printSchema()

In [ ]:
# Write to console for debugging
query = changes.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/bt_stream_checkpoint") \
    .trigger(processingTime="15 seconds") \
    .start()

print(f"Streaming query started: {query.id}")
print("Run query.stop() to stop the stream")

In [ ]:
# Stop the streaming query when done
# query.stop()

---
## Filter by Mutation Type

Process only specific mutation types (SET_CELL, DELETE_COLUMN, DELETE_FAMILY, DELETE_ROW).

In [ ]:
from pyspark.sql.functions import col

upserts = spark.readStream \
    .format("bigtable_changes") \
    .options(**common_options) \
    .load() \
    .filter(col("mutation_type") == "SET_CELL")

query = upserts.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/bt_upserts_checkpoint") \
    .trigger(processingTime="15 seconds") \
    .start()

print(f"Filtering SET_CELL mutations: {query.id}")

In [ ]:
# query.stop()

---
## Write to Delta Lake

Persist change stream events to a Delta table for downstream analytics.

In [ ]:
changes = spark.readStream \
    .format("bigtable_changes") \
    .options(**common_options) \
    .option("max_rows_per_partition", "10000") \
    .load()

query = changes.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "gs://my-bucket/checkpoints/bt-stream") \
    .trigger(processingTime="15 seconds") \
    .start("gs://my-bucket/delta/bigtable-changes")

print(f"Writing to Delta Lake: {query.id}")

In [ ]:
# query.stop()

---
## Watermark Support

Use the `low_watermark` column with Spark's `withWatermark()` for late data handling.

In [ ]:
from pyspark.sql.functions import col, window

changes = spark.readStream \
    .format("bigtable_changes") \
    .options(**common_options) \
    .option("batch_duration_seconds", "15") \
    .load()

windowed = changes \
    .withWatermark("commit_timestamp", "1 minute") \
    .groupBy(
        window(col("commit_timestamp"), "30 seconds"),
        col("mutation_type")
    ) \
    .count()

query = windowed.writeStream \
    .format("console") \
    .outputMode("update") \
    .option("checkpointLocation", "/tmp/bt_watermark_checkpoint") \
    .trigger(processingTime="30 seconds") \
    .start()

print(f"Windowed aggregation: {query.id}")

In [ ]:
# query.stop()

---
## Custom App Profile

Use a specific Bigtable app profile for the change stream reader.

In [ ]:
changes = spark.readStream \
    .format("bigtable_changes") \
    .options(**common_options) \
    .option("app_profile_id", "streaming-profile") \
    .option("batch_duration_seconds", "20") \
    .option("max_rows_per_partition", "8000") \
    .load()

query = changes.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/bt_profile_checkpoint") \
    .start()

print(f"Reading with custom app profile: {query.id}")

In [ ]:
# query.stop()

---
## Reconstruct full records with transformWithState

Consume the change stream and reconstruct the **latest row state** per `row_key`: the stateful processor keeps a map of **column_family → latest value** and emits one row `(row_key, record)` on every change. Requires Spark 4.x with RocksDB state store (e.g. Databricks Runtime with transformWithState support).

In [ ]:
# Set RocksDB state store (required for transformWithState)
spark.conf.set(
    "spark.sql.streaming.stateStore.providerClass",
    "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider",
)

from bigtable_stateful_processor import BigtableReconstructProcessor, RECONSTRUCTED_RECORD_SCHEMA

changes = spark.readStream.format("bigtable_changes").options(**common_options).load()

reconstructed = (
    changes.groupBy("row_key")
    .transformWithState(
        statefulProcessor=BigtableReconstructProcessor(),
        outputStructType=RECONSTRUCTED_RECORD_SCHEMA,
        outputMode="Update",
        timeMode="None",
    )
)

# Write to memory table; each row has row_key (binary) and record (map: column_family -> value bytes)
query = reconstructed.writeStream \
    .format("memory") \
    .queryName("bt_reconstructed") \
    .outputMode("update") \
    .option("checkpointLocation", "/tmp/bt_reconstruct_checkpoint") \
    .trigger(processingTime="10 seconds") \
    .start()

print(f"Reconstructing full records: {query.id}")
print("Query the in-memory table: spark.table('bt_reconstructed').show(20, truncate=60)")

In [ ]:
# query.stop()
# spark.table("bt_reconstructed").show(20, truncate=60)

---
## Configuration Reference

| Option | Required | Default | Description |
|--------|----------|---------|-------------|
| `project_id` | Yes | — | GCP project ID |
| `instance_id` | Yes | — | Bigtable instance ID |
| `table_id` | Yes | — | Bigtable table ID |
| `app_profile_id` | No | `default` | Bigtable app profile ID |
| `batch_duration_seconds` | No | `10` | Target duration per micro-batch |
| `max_rows_per_partition` | No | `5000` | Max mutations per partition per batch |